# Replication walkthrough

Runs the Python replication pipeline step by step on the synthetic panel (or on the real panel if `data/real/huafeng_panel_person_wave.csv` is present). Each step corresponds to a script in `code/` and to a thesis location given in `README.md`.

Run from the repository root: `jupyter lab notebooks/01_replication_walkthrough.ipynb`.

In [ ]:
import os, sys, subprocess, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
CODE = ROOT / 'code'
sys.path.insert(0, str(CODE))
def run(script):
    r = subprocess.run([sys.executable, str(CODE / script)], capture_output=True, text=True, cwd=CODE)
    print(r.stdout); print(r.stderr[-2000:] if r.returncode else '')
print('Repository root:', ROOT)

## 0. Synthetic data
Generates `data/synthetic/*` from the parameters published in the thesis (Tables 4-4, 5-1, 5-2, 5-3, 5-13; §4.9.3) and prints achieved vs target moments.

In [ ]:
run('00_generate_synthetic_data.py')

## 1. Preprocessing and quality checks (Appendix C.1; §4.4.3, §5.4)

In [ ]:
run('01_data_preprocessing.py')

## 2. Measurement model checks (§4.5.7, §4.7.1, §4.9.3, §5.4, Table D-1)
The CFA itself is run in Mplus (`mplus/01_cfa_measurement.inp`); this step reproduces α, CR/AVE approximations, HTMT, EFA eigenvalues, KMO/Bartlett, parallel analysis, ICC and the Harman / marker-variable CMB checks.

In [ ]:
run('02_measurement_checks.py')

## 3. Structural paths — composite cross-check of Table 5-3 (Appendix C.2, D.2)

In [ ]:
run('03_structural_paths_crosscheck.py')

## 4. Bootstrap mediation (Appendix C.3; Tables 5-4, 5-16)
statsmodels `Mediation` as printed in the thesis, plus an employee-cluster bootstrap with percentile and bias-corrected intervals.

In [ ]:
run('04_bootstrap_mediation.py')

## 5. Instrumental variables (Appendix C.4; Table 5-3 Panel B)

In [ ]:
run('05_iv_estimation.py')

## 6. Moderation and interaction (H8–H10; Tables 5-6, 5-7, 5-13)

In [ ]:
run('06_moderation_interaction.py')

## 7. Robustness (Appendix C.5; Tables 5-5, 5-8 to 5-12, D-3, D-4)

In [ ]:
run('07_robustness_checks.py')

## 8. MMDV valuation worksheet (Appendix E; Tables 5-14, 5-15)

In [ ]:
run('08_mmdv_worksheet.py')

## 9. Export data for Mplus
Writes `mplus/panel.dat` and the matching `NAMES` statement; then run the `.inp` files in Mplus 8.8.

In [ ]:
run('09_prepare_mplus_data.py')

## Inspect result tables

In [ ]:
import pandas as pd
for f in sorted((ROOT / 'outputs').glob('0*.csv')):
    print('\n==', f.name); display(pd.read_csv(f).round(3))